In [1]:
!pip install gensim tensorflow

In [2]:
import warnings; warnings.filterwarnings("ignore")
import random, numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, matthews_corrcoef,
                             roc_auc_score, average_precision_score)
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D, GaussianNoise,
    SpatialDropout1D, Layer, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from gensim.models import Word2Vec, FastText

In [3]:
DATA_PATH    = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences.csv"
SEEDS        = [0, 1, 2]          # start with 3 for a first pass; use [0,1,2,3,4] for the paper
VEC_SIZE     = 100               # Word2Vec / FastText embedding size
CNN_ENCODERS = ["6mer", "TF-IDF-6mer", "word2vec6mer",
                "Word2Vec-TF-IDF-6mer", "CGR-TFIDF-6mer"]  # focused set; extend if you want
EPOCHS       = 120
BATCH        = 48

In [4]:
# ===== A-CLEANUP: build a clean, defensible class set -> prepared_sequences_clean.csv
import pandas as pd
df = pd.read_csv(DATA_PATH)

# KEEP only well-defined classes. "-" (missing label) and ad-hoc recombinant
# labels (A1D, 01B, BC, A1C, BF1, A1CD, 0107, 01BC) are excluded because they
# are not clean subtypes/established CRFs. Edit this list if you disagree.
KEEP = ["B", "C", "A1", "A6", "D", "G",          # pure subtypes / sub-subtypes
        "01_AE", "02_AG", "07_BC", "08_BC"]      # established CRFs

before = len(df)
df = df[df["subtype"].isin(KEEP)].copy()

# OPTIONAL: your shortest sequence is 1518 nt (partial pol). To keep near-full
# pol only, uncomment the next line:
# df = df[df["sequence"].str.len() >= 2500].copy()

df.to_csv("prepared_sequences_clean.csv", index=False)     # <- point Script B's DATA_PATH here
print(f"kept {len(df)}/{before} seqs | {df['subtype'].nunique()} classes | "
      f"{df['cluster'].nunique()} clusters")
print(df["subtype"].value_counts().to_string())

kept 24072/26292 seqs | 10 classes | 11295 clusters
subtype
B        13142
01_AE     4105
C         4012
A1         797
07_BC      748
D          352
08_BC      289
02_AG      272
A6         238
G          117


In [5]:
def kmers(seq, k): return [seq[i:i+k] for i in range(len(seq) - k + 1)]

def make_kmer_encoder(train, k):                       # L1-normalised k-mer frequency
    vocab = sorted({km for s in train for km in kmers(s, k)})
    idx = {kk: i for i, kk in enumerate(vocab)}
    def tf_(seqs):
        out = np.zeros((len(seqs), len(vocab)), np.float32)
        for i, s in enumerate(seqs):
            for km in kmers(s, k):
                if km in idx: out[i, idx[km]] += 1
            tot = out[i].sum()
            if tot > 0: out[i] /= tot                  # divide by length -> removes length bias
        return out
    return tf_

def make_cgr_encoder(max_len=256):                     # Chaos Game Representation walk
    m = {"A": (0, 0), "C": (0, 1), "G": (1, 1), "T": (1, 0)}
    def tf_(seqs):
        enc = []
        for s in seqs:
            x, y, path = 0.5, 0.5, []
            for c in s:
                if c in m:
                    dx, dy = m[c]; x = (x + dx) / 2; y = (y + dy) / 2; path.append(x + y)
            v = np.array(path, np.float32)
            v = v[:max_len] if len(v) >= max_len else np.pad(v, (0, max_len - len(v)))
            enc.append(v)
        return np.array(enc, np.float32)
    return tf_

def make_w2v_encoder(train, k, vs=VEC_SIZE):           # averaged Word2Vec of k-mers
    mdl = Word2Vec([kmers(s, k) for s in train], vector_size=vs, window=5,
                   min_count=1, workers=4, seed=42, epochs=10)
    def tf_(seqs):
        out = []
        for s in seqs:
            v = [mdl.wv[w] for w in kmers(s, k) if w in mdl.wv]
            out.append(np.mean(v, 0) if v else np.zeros(vs))
        return np.array(out, np.float32)
    return tf_

def make_ft_encoder(train, k, vs=VEC_SIZE):            # averaged FastText of k-mers
    mdl = FastText([kmers(s, k) for s in train], vector_size=vs, window=5,
                   min_count=1, workers=4, seed=42, epochs=10)
    def tf_(seqs):
        out = []
        for s in seqs:
            v = [mdl.wv[w] for w in kmers(s, k)]
            out.append(np.mean(v, 0) if v else np.zeros(vs))
        return np.array(out, np.float32)
    return tf_

def make_tfidf_encoder(train, k, max_features=512):    # TF-IDF over k-mer "words"
    vec = TfidfVectorizer(max_features=max_features)
    vec.fit([" ".join(kmers(s, k)) for s in train])
    return lambda seqs: vec.transform(
        [" ".join(kmers(s, k)) for s in seqs]).toarray().astype(np.float32)

def make_w2v_tfidf_encoder(train, k):                  # hybrid: concat W2V + TF-IDF
    w = make_w2v_encoder(train, k); t = make_tfidf_encoder(train, k)
    return lambda seqs: np.concatenate([w(seqs), t(seqs)], axis=1)

def make_cgr_tfidf_encoder(train, k):                  # hybrid: concat CGR + TF-IDF
    c = make_cgr_encoder(); t = make_tfidf_encoder(train, k)
    return lambda seqs: np.concatenate([c(seqs), t(seqs)], axis=1)



In [6]:
def build_encoders(X_tr, names):
    # A menu of all encoders. Only the ones you list get built (each build costs time).
    reg = {
        "6mer":                 lambda: make_kmer_encoder(X_tr, 6),
        "4mer":                 lambda: make_kmer_encoder(X_tr, 4),
        "TF-IDF-6mer":          lambda: make_tfidf_encoder(X_tr, 6),
        "TF-IDF-4mer":          lambda: make_tfidf_encoder(X_tr, 4),
        "word2vec6mer":         lambda: make_w2v_encoder(X_tr, 6),
        "word2vec4mer":         lambda: make_w2v_encoder(X_tr, 4),
        "fasttext6mer":         lambda: make_ft_encoder(X_tr, 6),
        "cgr6mer":              lambda: make_cgr_encoder(),
        "Word2Vec-TF-IDF-6mer": lambda: make_w2v_tfidf_encoder(X_tr, 6),
        "CGR-TFIDF-6mer":       lambda: make_cgr_tfidf_encoder(X_tr, 6),
    }
    return {n: reg[n]() for n in names}

In [7]:
# --- Inception-v1 (1D) — GoogLeNet-style parallel 1/3/5-wide branches ----------
def inception_module(x, f1, f3r, f3, f5r, f5, fp):
    b1 = Conv1D(f1, 1, padding="same", activation="relu")(x)                 # 1-wide branch
    b3 = Conv1D(f3, 3, padding="same", activation="relu")(                    # 3-wide (1x1 reduce first)
             Conv1D(f3r, 1, padding="same", activation="relu")(x))
    b5 = Conv1D(f5, 5, padding="same", activation="relu")(                    # 5-wide (1x1 reduce first)
             Conv1D(f5r, 1, padding="same", activation="relu")(x))
    bp = Conv1D(fp, 1, padding="same", activation="relu")(                    # pooling branch
             MaxPooling1D(3, strides=1, padding="same")(x))
    return Concatenate()([b1, b3, b5, bp])                                    # stack all branches

def build_inception(input_len, n_classes):
    inp = Input((input_len, 1))
    x = MaxPooling1D(3, strides=2, padding="same")(
            ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same")(inp))))
    x = inception_module(x, 16, 24, 32, 4,  8,  8)
    x = inception_module(x, 32, 32, 48, 8, 24, 16)
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 48, 24, 52, 8, 24, 16)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4),
              loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

ARCHITECTURES = {"inception": build_inception}

In [8]:
# ===== CELL 3: leakage-free split (grouped by cluster, stratified by class) =======
def grouped_split(X, y, groups, seed):
    # THE FIX: StratifiedGroupKFold keeps every cluster wholly inside ONE set
    # (so near-duplicate sequences can't straddle train/test) AND balances class
    # proportions across the split.
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(sgkf.split(X, y, groups))                 # fold 0's test = ~20% -> test
    sgkf2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(sgkf2.split(X[trv], y[trv], groups[trv]))   # carve val from train
    return trv[tr_rel], trv[va_rel], te

In [9]:
# ===== CELL 4: metrics helper (safe for any number of classes) ====================
def score(y_true, y_pred, y_prob=None, n_classes=2):
    d = dict(acc=accuracy_score(y_true, y_pred),
             bal_acc=balanced_accuracy_score(y_true, y_pred),
             f1=f1_score(y_true, y_pred, average="macro"),
             kappa=cohen_kappa_score(y_true, y_pred),
             mcc=matthews_corrcoef(y_true, y_pred))          # MCC stays honest under imbalance
    if y_prob is not None:
        try:                                                 # probability-based metrics (CNNs only)
            oh = to_categorical(y_true, n_classes)
            d["auroc"] = roc_auc_score(oh, y_prob, multi_class="ovr", average="macro")
            d["auprc"] = average_precision_score(oh, y_prob, average="macro")
        except Exception:
            d["auroc"] = d["auprc"] = np.nan
    return {k: round(float(v), 4) for k, v in d.items()}

In [10]:
# ===== CELL 5: main loop — for each seed: split, encode once, run baselines + 3 CNNs ==
df = pd.read_csv("prepared_sequences_clean.csv")
X_all = df["sequence"].values
grp   = df["cluster"].values
le    = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
n_cls = len(le.classes_)
print(f"{len(df)} seqs | {n_cls} classes {list(le.classes_)} | "
      f"majority-class baseline acc = {np.bincount(y_all).max()/len(y_all):.3f}")

base_rows, cnn_rows = [], []
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = grouped_split(X_all, y_all, grp, seed)
    X_tr, X_va, X_te = X_all[tr], X_all[va], X_all[te]
    y_tr, y_va, y_te = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)))
    print(f"\n[seed {seed}] train={len(tr)} val={len(va)} test={len(te)}")

    # build every encoder we need this seed (TF-IDF-6mer always, for the baselines)
    encoders = build_encoders(X_tr, sorted(set(CNN_ENCODERS) | {"TF-IDF-6mer"}))

    # ---- baselines on TF-IDF-6mer (the "does deep learning even help?" reference) ----
    Btr, Bte = encoders["TF-IDF-6mer"](X_tr), encoders["TF-IDF-6mer"](X_te)
    for name, clf in [("LogReg", LogisticRegression(max_iter=2000, class_weight="balanced")),
                      ("LinearSVM", LinearSVC(class_weight="balanced")),
                      ("RandomForest", RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                       class_weight="balanced", random_state=seed))]:
        clf.fit(Btr, y_tr)
        base_rows.append({"seed": seed, "model": name,
                          **score(y_te, clf.predict(Bte), None, n_cls)})

    # ---- CNNs: encode ONCE per encoding, then train all three architectures on it ----
    for enc_name in CNN_ENCODERS:
        enc = encoders[enc_name]
        sc  = StandardScaler()
        Xtr = sc.fit_transform(enc(X_tr))[..., None]        # scaler fitted on train only
        Xva = sc.transform(enc(X_va))[..., None]
        Xte = sc.transform(enc(X_te))[..., None]
        for arch_name, build_fn in ARCHITECTURES.items():
            model = build_fn(Xtr.shape[1], n_cls)
            model.fit(Xtr, to_categorical(y_tr, n_cls),
                      validation_data=(Xva, to_categorical(y_va, n_cls)),
                      epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
                      callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                                 ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
            prob = model.predict(Xte, verbose=0)
            cnn_rows.append({"seed": seed, "architecture": arch_name, "encoding": enc_name,
                             **score(y_te, prob.argmax(1), prob, n_cls)})
            print(f"  {arch_name:10s} {enc_name:22s} "
                  f"acc={cnn_rows[-1]['acc']} f1={cnn_rows[-1]['f1']} mcc={cnn_rows[-1]['mcc']}")


24072 seqs | 10 classes ['01_AE', '02_AG', '07_BC', '08_BC', 'A1', 'A6', 'B', 'C', 'D', 'G'] | majority-class baseline acc = 0.546

[seed 0] train=14783 val=4726 test=4563


I0000 00:00:1783785352.415555      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783785352.418566      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1783785372.685571     191 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  inception  6mer                   acc=0.8391 f1=0.579 mcc=0.7756
  inception  TF-IDF-6mer            acc=0.9441 f1=0.869 mcc=0.917
  inception  word2vec6mer           acc=0.9858 f1=0.9656 mcc=0.9784
  inception  Word2Vec-TF-IDF-6mer   acc=0.9439 f1=0.8944 mcc=0.9175
  inception  CGR-TFIDF-6mer         acc=0.9226 f1=0.7994 mcc=0.8878

[seed 1] train=14686 val=4599 test=4787
  inception  6mer                   acc=0.8066 f1=0.5214 mcc=0.7014
  inception  TF-IDF-6mer            acc=0.9217 f1=0.8457 mcc=0.8826
  inception  word2vec6mer           acc=0.9896 f1=0.9673 mcc=0.9838
  inception  Word2Vec-TF-IDF-6mer   acc=0.94 f1=0.8443 mcc=0.9101
  inception  CGR-TFIDF-6mer         acc=0.981 f1=0.9251 mcc=0.9706

[seed 2] train=14088 val=5304 test=4680
  inception  6mer                   acc=0.8453 f1=0.6041 mcc=0.7769
  inception  TF-IDF-6mer            acc=0.9799 f1=0.9455 mcc=0.9685
  inception  word2vec6mer           acc=0.9865 f1=0.9603 mcc=0.9788
  inception  Word2Vec-TF-IDF-6mer   acc=

In [11]:
# ===== CELL 6 (Kaggle): aggregate to mean ± std, save to /kaggle/working ==========
import os
from IPython.display import display        # renders DataFrames as tidy HTML tables on Kaggle

OUT_DIR = "/kaggle/working"                 # <-- the ONLY folder Kaggle saves & lets you download
os.makedirs(OUT_DIR, exist_ok=True)

def summarise(rows, keys):
    """Group per-seed rows by `keys` and return mean & std of every metric.
       Then flatten the two-level (metric, mean/std) header into flat column
       names (e.g. 'acc_mean', 'acc_std') so the saved CSV is easy to read."""
    d = pd.DataFrame(rows)
    if isinstance(keys, str): keys = [keys]
    metrics = [c for c in d.columns if c not in (["seed"] + keys)]
    g = d.groupby(keys)[metrics].agg(["mean", "std"]).round(4)
    g.columns = [f"{metric}_{stat}" for metric, stat in g.columns]   # ('acc','mean') -> 'acc_mean'
    return g.reset_index()                                           # turn group keys into columns

# ---- build the four output tables -------------------------------------------------
base_raw     = pd.DataFrame(base_rows)                    # every seed × baseline model
cnn_raw      = pd.DataFrame(cnn_rows)                     # every seed × architecture × encoding
base_summary = summarise(base_rows, "model")             # baselines, mean ± std
cnn_summary  = summarise(cnn_rows, ["architecture", "encoding"])  # CNNs, mean ± std

# ---- save all four to /kaggle/working (appear in the Output tab) ------------------
paths = {
    "baseline_results_raw.csv":     base_raw,
    "baseline_results_summary.csv": base_summary,
    "cnn_results_raw.csv":          cnn_raw,
    "cnn_results_summary.csv":      cnn_summary,
}
for fname, table in paths.items():
    table.to_csv(os.path.join(OUT_DIR, fname), index=False)

# ---- show the summaries inline as rendered tables ---------------------------------
print("=== BASELINES (mean ± std over", len(SEEDS), "seeds) ===")
display(base_summary)
print("=== CNNs (mean ± std, by architecture × encoding) ===")
display(cnn_summary)

# ---- confirm what was written ------------------------------------------------------
print("\nSaved to /kaggle/working/ (see the Output tab to download or commit):")
for fname in paths:
    print("  •", fname)

=== BASELINES (mean ± std over 3 seeds) ===


,model,acc_mean,acc_std,bal_acc_mean,bal_acc_std,f1_mean,f1_std,kappa_mean,kappa_std,mcc_mean,mcc_std
0,LinearSVM,0.9971,0.0010,0.9904,0.0064,0.9898,0.0020,0.9955,0.0017,0.9955,0.0017
1,LogReg,0.9962,0.0013,0.9900,0.0067,0.9832,0.0039,0.9941,0.0021,0.9941,0.0021
2,RandomForest,0.9946,0.0013,0.9640,0.0011,0.9779,0.0030,0.9916,0.0022,0.9916,0.0022


=== CNNs (mean ± std, by architecture × encoding) ===


,architecture,encoding,acc_mean,acc_std,bal_acc_mean,bal_acc_std,f1_mean,f1_std,kappa_mean,kappa_std,mcc_mean,mcc_std,auroc_mean,auroc_std,auprc_mean,auprc_std
0,inception,6mer,0.8303,0.0208,0.6690,0.1408,0.5682,0.0424,0.7438,0.0458,0.7513,0.0432,0.9772,0.0068,0.6635,0.0580
1,inception,CGR-TFIDF-6mer,0.9594,0.0320,0.9347,0.0414,0.8870,0.0761,0.9388,0.0461,0.9395,0.0451,0.9975,0.0015,0.9613,0.0221
2,inception,TF-IDF-6mer,0.9486,0.0294,0.9379,0.0228,0.8867,0.0522,0.9221,0.0438,0.9227,0.0432,0.9969,0.0019,0.9536,0.0219
3,inception,Word2Vec-TF-IDF-6mer,0.9449,0.0055,0.9413,0.0042,0.8792,0.0303,0.9165,0.0073,0.9173,0.0072,0.9962,0.0004,0.9582,0.0038
4,inception,word2vec6mer,0.9873,0.0020,0.9691,0.0096,0.9644,0.0037,0.9803,0.0030,0.9803,0.0030,0.9994,0.0004,0.9914,0.0024



Saved to /kaggle/working/ (see the Output tab to download or commit):
  • baseline_results_raw.csv
  • baseline_results_summary.csv
  • cnn_results_raw.csv
  • cnn_results_summary.csv


In [12]:
!zip -r inceptionoutput.zip /kaggle/working/

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/baseline_results_raw.csv (deflated 58%)
  adding: kaggle/working/cnn_results_summary.csv (deflated 56%)
  adding: kaggle/working/baseline_results_summary.csv (deflated 53%)
  adding: kaggle/working/cnn_results_raw.csv (deflated 63%)
  adding: kaggle/working/prepared_sequences_clean.csv (deflated 94%)
  adding: kaggle/working/__notebook__.ipynb (deflated 78%)
